In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

/home/teaching/miniconda3/envs/dl45/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "models/gemma-2b-it"  # or your local path

'''
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
'''

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False              # 🔥 prevents memory issues
model.gradient_checkpointing_enable()      # 🔥 saves memory

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.33s/it]


In [3]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,194,880 || all params: 2,617,536,768 || trainable%: 0.12205673819211085


In [4]:
dataset = load_dataset("json", data_files={
    "train": "dataset/train.json",
    "validation": "dataset/val.json"
})

In [5]:
SYSTEM_INSTRUCTION = """
You are an AI system designed to transform raw, unstructured job descriptions into structured, professional, and ATS-friendly job descriptions.

Your task:
- Extract AND rewrite content into a polished, professional format.
- Expand short or vague statements into clear, detailed, and actionable bullet points.
- Improve grammar, clarity, and tone while preserving original meaning.

Rules:
- Output MUST be valid JSON only.
- Follow the exact schema provided.
- Do NOT include explanations or extra text.
- Do NOT hallucinate unrealistic details.
- Do NOT copy sentences directly from input — always rewrite them professionally.

Enhancement Rules:
- Convert short phrases into complete, professional sentences.
- Add clarity by specifying intent (e.g., “handle tickets” → “resolve IT support tickets efficiently within defined SLAs”).
- Use strong action verbs (e.g., manage, ensure, deliver, coordinate, analyze).
- Maintain ATS-friendly language with relevant keywords.
- Avoid vague wording like “do”, “work on”, “handle”.

Writing Style:
- Use concise but complete sentences.
- Each bullet point should be meaningful and self-contained.
- Maintain consistency across all sections.
"""

OUTPUT_SCHEMA = """
{
  "job_title": "",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
}
"""

def format_example(example):
  prompt =  f"""
### SYSTEM:
{SYSTEM_INSTRUCTION}

### USER:
Convert the following raw job description into structured JSON.

### Expected OUTPUT FORMAT:
Return a fully populated JSON following this schema:
{OUTPUT_SCHEMA}

### INPUT:
{example['input']}

"""
  full_text = prompt + "\n" + example["output"]
  return {"text": full_text}


In [7]:
dataset = dataset.map(format_example)

In [8]:
def tokenize(example):
    tokens =  tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 58/58 [00:00<00:00, 2136.62 examples/s]


In [10]:
training_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=1,   # 🔥 reduce
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,   # 🔥 simulate larger batch

    num_train_epochs=3,
    learning_rate=2e-4,

    fp16=True,

    logging_steps=10,
    save_steps=100,

    save_total_limit=2,
    report_to="none"
)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

In [12]:
trainer.train()

  6%|▌         | 10/171 [00:11<03:00,  1.12s/it]

{'loss': 1.8361, 'grad_norm': 1.3507249355316162, 'learning_rate': 0.00018830409356725147, 'epoch': 0.17}


 12%|█▏        | 20/171 [00:22<02:49,  1.12s/it]

{'loss': 0.9874, 'grad_norm': 1.1246455907821655, 'learning_rate': 0.00017660818713450294, 'epoch': 0.35}


 18%|█▊        | 30/171 [00:34<02:41,  1.15s/it]

...

 99%|█████████▉| 170/171 [03:15<00:01,  1.15s/it]

{'loss': 0.3219, 'grad_norm': 0.5613023638725281, 'learning_rate': 1.1695906432748538e-06, 'epoch': 2.94}


100%|██████████| 171/171 [03:17<00:00,  1.16s/it]

{'train_runtime': 197.6001, 'train_samples_per_second': 3.507, 'train_steps_per_second': 0.865, 'train_loss': 0.4852332807423776, 'epoch': 2.96}


TrainOutput(global_step=171, training_loss=0.4852332807423776, metrics={'train_runtime': 197.6001, 'train_samples_per_second': 3.507, 'train_steps_per_second': 0.865, 'total_flos': 4260727398334464.0, 'train_loss': 0.4852332807423776, 'epoch': 2.961038961038961})

In [ ]:
model.save_pretrained("./models/gemma-2b-it-fine-tuned")
tokenizer.save_pretrained("./models/gemma-2b-it-fine-tuned")

('./models/gemma-2b-it-fine-tuned/tokenizer_config.json',
 './models/gemma-2b-it-fine-tuned/special_tokens_map.json',
 './models/gemma-2b-it-fine-tuned/tokenizer.json')

: 